In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import os

# Load SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Folder path
folder_path = "datasets/transcripts"

# Get all text files
files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

print("Files found:", files)

In [7]:
def create_blocks(embeddings, block_size):
    blocks = []
    for i in range(0, len(embeddings), block_size):
        block = np.mean(embeddings[i:i+block_size], axis=0)
        blocks.append(block)
    return blocks

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def segment_text(sentences, block_size, k):
    embeddings = model.encode(sentences)
    blocks = create_blocks(embeddings, block_size)

    similarities = []
    for i in range(len(blocks)-1):
        sim = cosine_similarity(blocks[i], blocks[i+1])
        similarities.append(sim)

    similarities = np.array(similarities)
    threshold = similarities.mean() - k * similarities.std()

    boundaries = [i for i, sim in enumerate(similarities) if sim < threshold]

    return boundaries, similarities, threshold

In [8]:
def segment_text(sentences, block_size, k):
    embeddings = model.encode(sentences)

    # create blocks
    blocks = []
    for i in range(0, len(embeddings), block_size):
        block = np.mean(embeddings[i:i+block_size], axis=0)
        blocks.append(block)

    # similarity between blocks
    similarities = []
    for i in range(len(blocks)-1):
        sim = np.dot(blocks[i], blocks[i+1]) / (
            np.linalg.norm(blocks[i]) * np.linalg.norm(blocks[i+1])
        )
        similarities.append(sim)

    similarities = np.array(similarities)

    threshold = similarities.mean() - k * similarities.std()

    boundaries = [i for i, sim in enumerate(similarities) if sim < threshold]

    return boundaries, similarities, threshold

In [9]:
block_sizes = [3, 5, 8]
k_values = [0.8, 1.0, 1.2]

all_results = []

for file in files:
    file_path = os.path.join(folder_path, file)

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    sentences = [s.strip() for s in text.split("\n") if s.strip()]

    print(f"\nProcessing: {file}")
    print("Total sentences:", len(sentences))

    for b in block_sizes:
        for k in k_values:
            boundaries, sims, th = segment_text(sentences, b, k)
            segments = len(boundaries) + 1
            all_results.append([file, b, k, segments])


Processing: audio1.txt
Total sentences: 742

Processing: audio10.txt
Total sentences: 383

Processing: audio11.txt
Total sentences: 256

Processing: audio12.txt
Total sentences: 547

Processing: audio13.txt
Total sentences: 588

Processing: audio14.txt
Total sentences: 506

Processing: audio15.txt
Total sentences: 160

Processing: audio16.txt
Total sentences: 231

Processing: audio17.txt
Total sentences: 567

Processing: audio2.txt
Total sentences: 450

Processing: audio3.txt
Total sentences: 280

Processing: audio4.txt
Total sentences: 190

Processing: audio5.txt
Total sentences: 299

Processing: audio6.txt
Total sentences: 253

Processing: audio7.txt
Total sentences: 154

Processing: audio8.txt
Total sentences: 118

Processing: audio9.txt
Total sentences: 84


In [10]:
import pandas as pd

results_df = pd.DataFrame(
    all_results,
    columns=["File", "Block Size", "k", "Segments"]
)

results_df

,File,Block Size,k,Segments
0,audio1.txt,3,0.8,56
1,audio1.txt,3,1.0,42
2,audio1.txt,3,1.2,29
3,audio1.txt,5,0.8,36
4,audio1.txt,5,1.0,21
...,...,...,...,...
148,audio9.txt,5,1.0,3
149,audio9.txt,5,1.2,3
150,audio9.txt,8,0.8,3
151,audio9.txt,8,1.0,3


In [11]:
output_folder = "segmented_outputs"
os.makedirs(output_folder, exist_ok=True)

for file in files:
    file_path = os.path.join(folder_path, file)

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    sentences = [s.strip() for s in text.split("\n") if s.strip()]

    boundaries, sims, th = segment_text(sentences, 5, 1.0)

    start = 0
    output_text = ""

    for b in boundaries:
        end = (b+1)*5
        segment = sentences[start:end]
        output_text += " ".join(segment) + "\n\n"
        start = end

    output_text += " ".join(sentences[start:])

    out_path = os.path.join(output_folder, file)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(output_text)

print(" Segmented files saved in:", output_folder)

 Segmented files saved in: segmented_outputs


evaluating for  a 2 audio files

In [12]:
def evaluate_segmentation(predicted, actual):
    predicted = set(predicted)
    actual = set(actual)

    tp = len(predicted & actual)
    fp = len(predicted - actual)
    fn = len(actual - predicted)

    precision = tp / (tp + fp) if len(predicted) > 0 else 0
    recall = tp / (tp + fn) if len(actual) > 0 else 0

    return precision, recall

In [13]:
# define manual boundaries
actual_audio1 = [2, 5]
actual_audio2 = [3, 7]

results = []

block_sizes = [3,5,8]
k_values = [0.8,1.0,1.2]

files = ["audio1.txt","audio2.txt"]
manual_bounds = [actual_audio1, actual_audio2]

for file, actual in zip(files, manual_bounds):

    with open(f"datasets/transcripts/{file}", "r", encoding="utf-8") as f:
        text = f.read()

    sentences = [s.strip() for s in text.split("\n") if s.strip()]

    for b in block_sizes:
        for k in k_values:

            predicted, _, _ = segment_text(sentences, b, k)

            precision, recall = evaluate_segmentation(predicted, actual)

            results.append([file, b, k, precision, recall])

In [14]:
import pandas as pd

eval_df = pd.DataFrame(
    results,
    columns=["File","Block Size","k","Precision","Recall"]
)

eval_df

,File,Block Size,k,Precision,Recall
0,audio1.txt,3,0.8,0.000000,0.0
1,audio1.txt,3,1.0,0.000000,0.0
2,audio1.txt,3,1.2,0.000000,0.0
3,audio1.txt,5,0.8,0.057143,1.0
4,audio1.txt,5,1.0,0.050000,0.5
5,audio1.txt,5,1.2,0.071429,0.5
6,audio1.txt,8,0.8,0.047619,0.5
7,audio1.txt,8,1.0,0.062500,0.5
8,audio1.txt,8,1.2,0.000000,0.0
9,audio2.txt,3,0.8,0.000000,0.0


keyword extraction 

In [15]:

from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import json

In [16]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
kw_model = KeyBERT(model='all-MiniLM-L6-v2')

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 129.32it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 123.75it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
def segment_text(sentences, block_size=5, k=1.0):
    embeddings = sbert_model.encode(sentences)

    blocks = []
    for i in range(0, len(embeddings), block_size):
        blocks.append(np.mean(embeddings[i:i+block_size], axis=0))

    similarities = []
    for i in range(len(blocks)-1):
        sim = np.dot(blocks[i], blocks[i+1]) / (
            np.linalg.norm(blocks[i]) * np.linalg.norm(blocks[i+1])
        )
        similarities.append(sim)

    similarities = np.array(similarities)
    threshold = similarities.mean() - k * similarities.std()

    boundaries = [i for i, sim in enumerate(similarities) if sim < threshold]

    return boundaries

In [18]:
def extract_keywords(text, top_n=6):
    kws = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1,2),
        stop_words='english',
        top_n=top_n
    )
    return [k[0] for k in kws]

In [19]:
input_folder = "datasets/transcripts"
output_folder = "segment_json"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

for file in files:
    print("Processing:", file)

    with open(os.path.join(input_folder, file), "r", encoding="utf-8") as f:
        text = f.read()

    sentences = [s.strip() for s in text.split("\n") if s.strip()]

    boundaries = segment_text(sentences)

    segments_json = []
    start_idx = 0
    segment_id = 1

    for b in boundaries:
        end_idx = (b+1)*5 - 1

        segment_text_block = " ".join(sentences[start_idx:end_idx+1])

        keywords = extract_keywords(segment_text_block, top_n=6)

        segments_json.append({
            "segment_id": segment_id,
            "start": start_idx,
            "end": end_idx,
            "text": segment_text_block,
            "keywords": keywords
        })

        start_idx = end_idx + 1
        segment_id += 1

    # last segment
    segment_text_block = " ".join(sentences[start_idx:])
    keywords = extract_keywords(segment_text_block, top_n=6)

    segments_json.append({
        "segment_id": segment_id,
        "start": start_idx,
        "end": len(sentences)-1,
        "text": segment_text_block,
        "keywords": keywords
    })

    # save JSON
    output_path = os.path.join(output_folder, file.replace(".txt", ".json"))
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(segments_json, f, indent=4)

print("\nSegment JSON files created!")

Processing: audio1.txt
Processing: audio10.txt
Processing: audio11.txt
Processing: audio12.txt
Processing: audio13.txt
Processing: audio14.txt
Processing: audio15.txt
Processing: audio16.txt
Processing: audio17.txt
Processing: audio2.txt
Processing: audio3.txt
Processing: audio4.txt
Processing: audio5.txt
Processing: audio6.txt
Processing: audio7.txt
Processing: audio8.txt
Processing: audio9.txt

Segment JSON files created!


summary

In [20]:
from transformers import pipeline

summarizer = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

C:\Users\ASHOKA MS\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASHOKA MS\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|███████████████████████████| 190/190 [00:00<00:00, 231.28it/s, 

In [3]:
def summarize_segment(text):

    # If segment is short, return as-is
    if len(text.split()) < 40:
        return text

    prompt = f"Summarize the following text in 2-3 sentences:\n{text}"

    output = summarizer(
        prompt,
        max_new_tokens=80,
        do_sample=False,
        return_full_text=False  # important
    )

    summary = output[0]["generated_text"].strip()

    # Safety fallback
    if not summary:
        summary = text[:200]

    return "In this segment, the speaker discusses" + summary.lower()

In [4]:
block_size = 5  # make sure this matches segmentation

segments_json = []
start_idx = 0
segment_id = 1
total_sentences = len(sentences)

for b in boundaries:

    end_idx = min((b + 1) * block_size - 1, total_sentences - 1)

    # Skip invalid segments
    if start_idx > end_idx:
        continue

    segment_text_block = " ".join(sentences[start_idx:end_idx + 1])

    keywords = extract_keywords(segment_text_block, top_n=6)
    summary = summarize_segment(segment_text_block)

    segments_json.append({
        "segment_id": segment_id,
        "start": start_idx,
        "end": end_idx,
        "text": segment_text_block,
        "keywords": keywords,
        "summary": summary
    })

    start_idx = end_idx + 1
    segment_id += 1


#  Add Final Segment (VERY IMPORTANT)

if start_idx < total_sentences:

    segment_text_block = " ".join(sentences[start_idx:])

    keywords = extract_keywords(segment_text_block, top_n=6)
    summary = summarize_segment(segment_text_block)

    segments_json.append({
        "segment_id": segment_id,
        "start": start_idx,
        "end": total_sentences - 1,
        "text": segment_text_block,
        "keywords": keywords,
        "summary": summary
    })

NameError: name 'sentences' is not defined